# GP_ELITE — cinq étapes

**Vous avez un tableau de mesures et vous voulez la formule, pas une prédiction.**
Ce notebook part de zéro et se termine sur *vos* données.

Rien à installer sur votre machine : tout tourne dans le navigateur.
Exécutez les cellules dans l'ordre avec **Maj + Entrée**.
*[🇬🇧 English version](https://colab.research.google.com/github/ariel95500-create/gp-elite/blob/main/examples/quickstart.ipynb)*


---


## 1. Installation

Trois dépendances, aucun compilateur, aucun second langage : c'est le point de
départ de tout le reste.


In [ ]:
!pip install -q gp-elite

import gp_elite
print('GP_ELITE', gp_elite.__version__, 'prêt.')


## 2. Votre première loi : Kepler, en huit points

Distance au Soleil et période orbitale des huit planètes. Rien d'autre.

Kepler a mis des années à établir cette relation. Voyons ce que la machine
en fait avec huit lignes de données.


In [ ]:
import numpy as np
from gp_elite import symbolic_regression

distance = np.array([0.387, 0.723, 1.000, 1.524, 5.203, 9.537, 19.191, 30.069])
periode  = np.array([0.241, 0.615, 1.000, 1.881, 11.862, 29.457, 84.011, 164.79])

resultat = symbolic_regression(
    distance.reshape(-1, 1), periode,
    feature_names=['distance'],
    operators='physical', generations=30, speed='fast', seed=0,
)
print(resultat.expression)


Vous devriez lire `0.00279171 + 0.999396 * distance * sqrt(distance)`,
c'est-à-dire **distance^1.5** — la troisième loi de Kepler.

La formule est écrite dans vos unités : en unités astronomiques et en années,
la constante de Kepler vaut 1, et c'est 0.9994 qui revient.


## 3. Les unités physiques : ce que les autres ne font pas

Ici, une expérience de ressort. On mesure un allongement en mètres et une
force en newtons, et on cherche la loi de Hooke `F = k·x`.

La raideur `k` **n'est pas dans les données** — c'est une constante physique
inconnue, et elle porte des unités. On demande au moteur de la déduire.


In [ ]:
from gp_elite import GPEliteRegressor

rng = np.random.RandomState(0)
allongement = rng.uniform(0.01, 0.10, 150).reshape(-1, 1)   # mètres
force = 250.0 * allongement[:, 0]                            # newtons

est = GPEliteRegressor(
    units=['m'],              # unité de la colonne d'entrée
    target_units='N',         # unité de la cible
    unknown_constant=True,    # la constante manquante est à déduire
    generations=25, speed='fast', random_state=0,
)
est.fit(allongement, force)

print('unités déduites :', est.constant_units_string())
print('valeur déduite  :', round(est.constant_value_, 4))


`[kg / s^2]` et `250.0` — soit exactement des newtons par mètre, et la
raideur exacte du ressort.

Le moteur ne s'est pas contenté de trouver la forme de la loi : il a annoncé
**les unités et la valeur d'une grandeur physique absente des données**.

Sans `units=`, ce problème est hors d'atteinte — aucune constante sans
dimension ne peut relier des mètres à des newtons.


---

## 4. Des données réelles : ce que disent des mesures de 1933

Tout ce qui précède était propre. Ici, non.

Au début des années 1930, Johann Nikuradse a collé des grains de sable de taille connue
à l'intérieur de tuyaux et mesuré le frottement de l'écoulement turbulent : **362 mesures
réelles**, six rapports de rugosité, des nombres de Reynolds d'environ 4 300 à 1 000 000.
Ces données servent encore aujourd'hui : la dépendance complète du frottement au nombre de
Reynolds et à la rugosité est considérée comme un problème ouvert.

Le moteur reçoit les nombres bruts et rien d'autre : le rapport de rugosité `r_k`, le
logarithme décimal du nombre de Reynolds, et le frottement sous la forme `log10(100*lambda)`.

La référence à battre est la relation de manuel de **Prandtl–von Kármán**, qui s'écrit
dans cet espace `2 - 2*log10(2*log10(r/k) + 1.74)`, sans aucun paramètre ajusté.


In [ ]:
import pandas as pd
import numpy as np

URL = ("https://github.com/EpistasisLab/pmlb/raw/master/datasets/"
       "nikuradse_1/nikuradse_1.tsv.gz")
df = pd.read_csv(URL, sep="\t", compression="gzip")

Xn = df[["r_k", "log_Re"]].to_numpy(float)
yn = df["target"].to_numpy(float)

def prandtl_von_karman(r_k):
    return 2.0 - 2.0 * np.log10(2.0 * np.log10(r_k) + 1.74)

print(f"{len(yn)} mesures, rapports de rugosité {sorted(df.r_k.unique())}")
r2_pvk = 1 - np.mean((prandtl_von_karman(Xn[:, 0]) - yn)**2) / np.var(yn)
print(f"Prandtl-von Karman, aucun paramètre ajusté : R2 = {r2_pvk:.4f}")


Maintenant la recherche. **Cette cellule prend environ une demi-minute** : des données
réelles n'ont pas de solution exacte où s'arrêter, le moteur utilise donc tout son budget.

Avec le protocole complet (`benchmarks/real_nikuradse.py` : `generations=30, restarts=4`,
lignes tenues à l'écart), le front de Pareto commence par une expression de six nœuds,
`1.10 - 0.134 * log(r_k)`, qui obtient R² 0.9357 sur les lignes tenues à l'écart, contre
0.9456 pour la loi de référence sur les mêmes lignes (0.9434 sur les 362 lignes, comme
affiché ci-dessus), et à partir de 25 nœuds il contient des formes qui font mieux (R² 0.9462
à 25 nœuds) sans ressembler à une loi. Le champion rendu va plus loin (R² 0.9886) avec une
expression de 53 nœuds que personne n'appellerait une loi : sur des données réelles, c'est
dans le front que se trouve la réponse lisible.
L'essai plus court ci-dessous affiche son propre front (sur notre machine : la même forme de
six nœuds d'abord, puis à sept nœuds `1.33 - 0.56*log(log(r_k))`, de la famille
logarithme-d'un-logarithme de la loi de référence) ; le tableau vous dit honnêtement ce qu'il
a trouvé, quoi que ce soit.
Sur notre machine, l'essai affiche aussi un avertissement : le champion rendu (53 nœuds)
élève `r_k` à la puissance `log_Re`, ce qui dépasse un million sur 304 des 362 lignes, et le
moteur plafonne une telle puissance ; la formule imprimée s'écarte donc de `predict()` sur
304 lignes elle aussi. Le moteur le dit plutôt que de vous remettre une formule qui ne
correspond pas à ses prédictions. Le R² du tableau est celui de `predict()`, et les formes
courtes en tête du front s'impriment exactement.


In [ ]:
import time
import numpy as np
from gp_elite import symbolic_regression

t0 = time.time()
m = symbolic_regression(Xn, yn, feature_names=['r_k', 'log_Re'],
                        operators='physical', normalize='none',
                        generations=30, speed='fast', restarts=2, seed=0,
                        parallel=False)   # le même résultat sur toutes les machines
print(f'recherche : {time.time()-t0:.0f} s\n')

def r2(e):
    return 1 - np.mean((e.predict(Xn) - yn)**2) / np.var(yn)

front = sorted({(int(e.size), round(r2(e), 6)): e
                for e in list(m.pareto or []) + [m]}.items())
print('taille R2      expression')
for (s, r), e in front:
    print(f'{s:>4}  {r:>6.4f}  {e.expression[:78]}')

bonnes = [(k, e) for k, e in front if k[1] > 0.90]
if bonnes:
    (s, r), e = bonnes[0]
    six = np.array(sorted(df.r_k.unique()))
    grille = np.column_stack([six, np.full(len(six), Xn[:, 1].mean())])
    pred, theorie = e.predict(grille), prandtl_von_karman(six)
    print(f'\nexpression la plus compacte au-dessus de R2=0.90 ({s} nœuds) :\n  {e.expression}')
    print(f"\n{'r/k':>7}{'Prandtl-von Karman':>21}{'GP_ELITE':>12}")
    for a, b, c in zip(six, theorie, pred):
        print(f'{a:>7.1f}{b:>21.4f}{c:>12.4f}')
    print(f'\ncorrélation avec la loi de manuel : '
          f'{np.corrcoef(theorie, pred)[0, 1]:.4f}')
else:
    print('\nrien de compact au-dessus de R2=0.90 dans cet essai court - voir '
          'benchmarks/real_nikuradse.py pour le protocole complet')


**Ce que cela montre, et ce que cela ne montre pas.** GP_ELITE ne bat pas ici la loi
connue. Sur un rapport de rugosité retiré entièrement de l'entraînement (`r_k = 126`), la
relation de manuel prédit le niveau à 0,04 écart-type de la courbe près. Les treize formes que
GP_ELITE propose sans ce rapport vont de 0,01 à 1,12 écart-type. Deux formes longues (52 et 59
nœuds) tombent plus près que la loi, mais seul un choix fait après avoir vu la courbe retirée
les désignerait ; les formes courtes, celles qu'on peut lire, ratent de 0,22 écart-type ou plus.

C'est le résumé honnête, et c'est le propos : à partir des seules mesures brutes, le moteur
propose des formes qui ajustent, parfois dans la bonne famille, et un test sur des données
tenues à l'écart vous dit jusqu'où elles portent. Le protocole complet, le test de la
rugosité retirée et la télémétrie sont dans `benchmarks/real_nikuradse.py`.


## 5. À votre tour

Chargez un CSV : une colonne cible, une ou plusieurs colonnes d'entrée.
Cent à quelques milliers de lignes, jusqu'à une dizaine de variables.


In [ ]:
import pandas as pd, io

try:
    from google.colab import files
    envoi = files.upload()
    nom = list(envoi)[0]
    df = pd.read_csv(io.BytesIO(envoi[nom]))
except ImportError:
    df = pd.read_csv('mes_donnees.csv')   # hors Colab : votre chemin ici

print(df.shape, 'lignes x colonnes')
df.head()


In [ ]:
# Adaptez ces deux lignes à vos colonnes.
CIBLE    = df.columns[-1]
ENTREES  = [c for c in df.columns if c != CIBLE]

X = df[ENTREES].to_numpy(dtype=float)
y = df[CIBLE].to_numpy(dtype=float)

res = symbolic_regression(X, y, feature_names=ENTREES,
                          operators='physical', generations=40,
                          speed='fast', seed=0)
print(res.expression)
print('R² sur le hold-out interne :', res.r2_validation)
# Ce hold-out sert aussi à choisir le modèle rendu : ce score est optimiste.
# Gardez vos propres données de test hors de l'ajustement pour juger la formule.


> **Si l'une de vos colonnes est un angle**, ou alimente une exponentielle
> ou un logarithme, essayez aussi `normalize='none'` et comparez. Le moteur
> divise chaque colonne par sa plus grande valeur avant de chercher : sans
> effet sur un produit, mais dans une fonction périodique `cos(theta)` doit
> alors être trouvé sous la forme `cos(6.28 * theta')`. Sur le banc Feynman
> de cette version (un essai par équation), le réglage par défaut ne rend
> exacte qu'une des trois lois qui contiennent un sinus ou un cosinus
> (II.15.4) ; `normalize='none'` les rend exactes toutes les trois.


---

## Ça n'a pas marché ?

GP_ELITE est réglé sur des benchmarks publiés — propres, sans bruit, bien
échelonnés. Vos mesures ne sont rien de tout cela, et c'est précisément là
que le moteur a besoin de progresser.

Un échec sur vos données est une **information utile, pas une erreur de votre
part**. [Ouvrez une issue](https://github.com/ariel95500-create/gp-elite/issues/new/choose)
avec la forme de vos données et ce que vous avez obtenu — pas besoin de
partager les données, pas besoin de savoir pourquoi. Français ou anglais.

## Pour aller plus loin

- [README](https://github.com/ariel95500-create/gp-elite#readme) — mode robuste,
  front de Pareto, extrapolation, audit dimensionnel
- `gp-elite` en ligne de commande : une interface console sans écrire de Python
- [CHANGELOG](https://github.com/ariel95500-create/gp-elite/blob/main/CHANGELOG.md)
  — chaque affirmation du README est adossée à une mesure reproductible
